-load biasbios dataset
-test Genderlabeling
-test swap
(...)


In [1]:
from pathlib import Path
import os
import pandas as pd
os.chdir(os.path.pardir)
print(f"Project root: {Path.cwd()}")



Project root: c:\Users\lunaj\Desktop\gender-bias-hate-speech


In [2]:
from src.data.gender_labeling import gender_lexicon, add_gender_label

In [3]:


data_path = Path.cwd() / "data" / "raw" / "BiasBios" 
# Load the .plk (pickle) file
train_data = pd.read_pickle(data_path / "train.pkl")
dev_data = pd.read_pickle(data_path / "dev.pkl")
test_data = pd.read_pickle(data_path / "test.pkl")

# Convert to a DataFrame and combine to giant Test Set
frames = []
for split in ("train", "dev", "test"):
    data = pd.read_pickle(data_path / f"{split}.pkl")
    frame = pd.DataFrame(data)
    frame["source_split"] = split
    frames.append(frame)

df = pd.concat(frames, ignore_index=True)

print(f"Anzahl Beispiele: {len(df)}")
print(df.columns.tolist())
display(df.head())


Anzahl Beispiele: 393423
['g', 'p', 'text', 'start', 'hard_text', 'hard_text_untokenized', 'text_without_gender', 'source_split']


,g,p,text,start,hard_text,hard_text_untokenized,text_without_gender,source_split
0,f,teacher,Hawa Sabriye is an Occasional Teacher with the...,149,"She has been working with children in camp , c...","She has been working with children in camp, co...","_ has been working with children in camp, comm...",train
1,m,professor,Tomas Norton is a tenure-track assistant profe...,170,He holds a PhD in Biosystems Engineering from ...,He holds a PhD in Biosystems Engineering from ...,_ holds a PhD in Biosystems Engineering from U...,train
2,m,attorney,John Dawson is an attorney who retired after p...,120,John served as the Department Chair of Foley ’...,John served as the Department Chair of Foley’s...,_ served as the Department Chair of Foley’s na...,train
3,f,surgeon,Dr. Konstantia Stoforou is a Neurosurgeon who ...,159,She has more 20 years of experience in the fie...,She has more 20 years of experience in the fie...,_ has more 20 years of experience in the field...,train
4,m,photographer,Ash Wright is a Louisville photographer who sp...,129,Over the years Ash has built an impressive fas...,Over the years Ash has built an impressive fas...,Over the years _ has built an impressive fashi...,train


In [4]:

for col in ["text","hard_text","hard_text_untokenized","text_without_gender"]:
        df = df.apply(
            add_gender_label,
            axis=1,
            output_column_name=f"label_{col}",
            input_column_name=col,
    )


In [5]:
print(df.columns.tolist())
display(df.head())


['g', 'p', 'text', 'start', 'hard_text', 'hard_text_untokenized', 'text_without_gender', 'source_split', 'label_text', 'label_hard_text', 'label_hard_text_untokenized', 'label_text_without_gender']


,g,p,text,start,hard_text,hard_text_untokenized,text_without_gender,source_split,label_text,label_hard_text,label_hard_text_untokenized,label_text_without_gender
0,f,teacher,Hawa Sabriye is an Occasional Teacher with the...,149,"She has been working with children in camp , c...","She has been working with children in camp, co...","_ has been working with children in camp, comm...",train,1,1,1,3
1,m,professor,Tomas Norton is a tenure-track assistant profe...,170,He holds a PhD in Biosystems Engineering from ...,He holds a PhD in Biosystems Engineering from ...,_ holds a PhD in Biosystems Engineering from U...,train,0,0,0,3
2,m,attorney,John Dawson is an attorney who retired after p...,120,John served as the Department Chair of Foley ’...,John served as the Department Chair of Foley’s...,_ served as the Department Chair of Foley’s na...,train,0,0,0,3
3,f,surgeon,Dr. Konstantia Stoforou is a Neurosurgeon who ...,159,She has more 20 years of experience in the fie...,She has more 20 years of experience in the fie...,_ has more 20 years of experience in the field...,train,1,1,1,3
4,m,photographer,Ash Wright is a Louisville photographer who sp...,129,Over the years Ash has built an impressive fas...,Over the years Ash has built an impressive fas...,Over the years _ has built an impressive fashi...,train,0,0,0,3


In [6]:
label_names = {0: "male", 1: "female", 2: "equal", 3: "none"}

label_columns = [
    "label_text",
    "label_hard_text",
    "label_hard_text_untokenized",
    "label_text_without_gender",
]

expected_gender = df["g"].map({"m": 0, "f": 1})

# Unbekannte oder fehlende Referenzlabels nicht unbemerkt auswerten
if expected_gender.isna().any():
    raise ValueError("Spalte g enthält fehlende oder andere Werte als m/f.")

results = []

for col in label_columns:
    expected = (
        pd.Series(3, index=df.index)
        if col == "label_text_without_gender"
        else expected_gender
    )

    matches = df[col].eq(expected)

    results.append({
        "column": col,
        "n": len(df),
        "matches": int(matches.sum()),
        "mismatches": int((~matches).sum()),
        "agreement_percent": matches.mean() * 100,
        "equal": int(df[col].eq(2).sum()),
        "none": int(df[col].eq(3).sum()),
    })

summary = pd.DataFrame(results).set_index("column")
display(summary.round(2))

,n,matches,mismatches,agreement_percent,equal,none
column,,,,,,
label_text,393423,391202,2221,99.44,1821,76
label_hard_text,393423,387747,5676,98.56,1604,3741
label_hard_text_untokenized,393423,387746,5677,98.56,1605,3741
label_text_without_gender,393423,347829,45594,88.41,3644,347829


In [7]:
for col in label_columns:
    print(col)

    table = pd.crosstab(df["g"], df[col])
    table = (
        table
        .reindex(index=["m", "f"], columns=[0, 1, 2, 3], fill_value=0)
        .rename(columns=label_names)
    )

    display(table)

label_text


label_text,male,female,equal,none
g,,,,
m,210233,163,922,3
f,161,180969,899,73


label_hard_text


label_hard_text,male,female,equal,none
g,,,,
m,208680,159,834,1648
f,172,179067,770,2093


label_hard_text_untokenized


label_hard_text_untokenized,male,female,equal,none
g,,,,
m,208680,159,834,1648
f,172,179066,771,2093


label_text_without_gender


label_text_without_gender,male,female,equal,none
g,,,,
m,10549,7270,1534,191968
f,7140,16991,2110,155861


In [8]:
col = "label_hard_text_untokenized"
text_col = col.removeprefix("label_")

expected = (
    pd.Series(3, index=df.index)
    if col == "label_text_without_gender"
    else expected_gender
)

mismatch_mask = df[col].ne(expected)

display(
    df.loc[mismatch_mask, ["g", text_col, col]].head(20)
)

,g,hard_text_untokenized,label_hard_text_untokenized
131,m,He is 40 years old and has been practicing for...,2
181,m,"Their late mother, Mary Gates, was a schooltea...",1
202,m,For more information on working with a persona...,3
224,f,Mrs. Deborah Leigh Henry's NPI Number is #1790...,3
415,m,"Subjects to be found include: London scenes, l...",3
496,f,She studies the political economy of informati...,2
541,m,Tushar Bansal is 3rd Year Student with Departm...,3
557,f,She has covered Indiana University men s socce...,2
575,f,"Specifically, Iulo's work has been recognized ...",3
640,f,She is a Visiting Physician at Royal Prince Al...,0


In [9]:
from collections import Counter
from src.data.gender_labeling import get_words, gender_lexicon


def inspect_gender_results(df, text_col="text", case="wrong", n=10, seed=42):
    label_col = f"label_{text_col}"
    expected = df["g"].map({"m": 0, "f": 1})
    predicted = df[label_col]

    masks = {
        # Eindeutige Zuordnung zum entgegengesetzten Gender
        "wrong": predicted.isin([0, 1]) & predicted.ne(expected),

        # Gleich viele männliche und weibliche Treffer
        "equal": predicted.eq(2),

        # Keine männlichen oder weiblichen Treffer
        "none": predicted.eq(3),

        # Mindestens ein Gender-Begriff erkannt, auch bei Gleichstand
        "remaining": predicted.isin([0, 1, 2]),
    }

    if expected.isna().any():
        raise ValueError("Spalte g enthält fehlende oder unbekannte Werte.")

    if case not in masks:
        raise ValueError(f"case muss eines von {list(masks)} sein.")

    selected = df.loc[masks[case], ["g", text_col, label_col]]
    examples = selected.sample(
        n=min(n, len(selected)),
        random_state=seed,
    ).copy()

    def matched_terms(text, gender):
        return dict(Counter(
            word for word in get_words(text)
            if word in gender_lexicon[gender]
        ))

    examples["male_terms"] = examples[text_col].apply(
        lambda text: matched_terms(text, "male")
    )
    examples["female_terms"] = examples[text_col].apply(
        lambda text: matched_terms(text, "female")
    )

    print(f"{len(selected):,} passende Fälle; zeige {len(examples)}.")
    return examples

In [10]:
# Falsche männliche/weibliche Zuordnungen
examples = inspect_gender_results(df, "hard_text", case="wrong")

with pd.option_context("display.max_colwidth", None):
    display(examples)

#Namen werden nicht erkannt
#his,her
#talking about family members brings to many gender terms -> maybe term cap? making it more strict?

331 passende Fälle; zeige 10.


,g,hard_text,label_hard_text,male_terms,female_terms
23640,m,"His work has been published at the Guardian UK , Reuters , IDN - InDepthNews , , Equal Times , IRIN News , Women & Girls News Deeply , Inter Press Service ( IPS ) , Kenya London News ( KLN ) and Sports News Arena ( SNA ) .",1,{'his': 1},"{'women': 1, 'girls': 1}"
365346,f,"She graduated with honors in 2001 . Having more than 15 years of diverse experiences , especially in CLINICAL NURSE SPECIALIST , Donna S King affiliates with Summa Health System , and cooperates with other doctors and specialists in medical group Summa Physicians Inc. Call Donna S King on phone number ( 330 ) 375 - 5100 for more information and advises or to book an appointment .",0,{'king': 2},{'she': 1}
84152,m,"His work has been published in the Journal of Women 's History , Labour / Le Travail , and Reviews in History . Elena Poniatowska received a lifetime achievement award from the International Women 's Media Foundation and is the first woman to win Mexico 's National Journalism Prize .",1,{'his': 1},"{'women': 2, 'woman': 1}"
276815,f,She is in the news after dating fellow model Presley Gerber . Presley Gerber is the son of supermodel Cindy Crawford and businessman Rande Gerber . The pair is now riding high in their romance . They do n't back off from magnificent their love towards each other in social media . The popular Cayley King bio is given below .,0,"{'son': 1, 'businessman': 1, 'king': 1}",{'she': 1}
65091,f,"She is a member of the Michael King Writers ’ Centre Trust and co - facilitates the Michael King Young Writers ’ Programme with poet and teacher , Johanna Emeney .",0,{'king': 2},{'she': 1}
214866,f,"Monique has reported for The Weather Channel , KPIX - TV ( CBS ) , KNTV - TV ( NBC ) , KSEE-24 ( NBC ) , KGTV - TV ( ABC ) , KPVI - TV ( NBC ) and has worked as a producer and a radio broadcaster . Monique is an experienced emcee and parade host . She ’s hosted the live broadcas​ts for the ​​Livermore Holida​y Sights and Sounds Parade,​​Fresno Christmas Parade and Fresno Veterans Day Parade . Monique has a certification from the Court of Master Sommeliers , is an experienced wine judge , and has interviewed at least two hundred winemakers , dozens of top chefs , and top celebrities like Jennifer Lawrence , Nicole Kidman and Kate Hudson . Monique is also a trained SAG / AFTRA actor and can be seen in various films , television shows , plays and commercials .",0,"{'host': 1, 'actor': 1}",{'she': 1}
342211,f,"Estelle ’s self - funded work exposed an injustice which led to the 2002 and 2005 exonerations of two men convicted of Perth killings in the ' 60s - the longest - standing convictions to be overturned in Australia . Her unfunded , determined sleuthing unearthed fresh evidence that prompted the Attorney General to allow the men new appeals after they had lost a combined total of seven Appeals in the ' 60s .",0,{'men': 2},{'her': 1}
257046,m,"His NSFW photographs , which are mainly shot in black and white , follow the naked or semi - naked bodies of women , showing only the upper body or just the legs or sometimes the entire figure . The black and white along with the women ’s poses are making the images romantic and nobody could perceive them as pornographic even if they tried a lot .",1,{'his': 1},{'women': 2}
334437,m,"Motivated by the love that Pramukh Swami Maharaj has shown to others throughout his lifetime , the women leveraged samp and samjan in orchestrating the performance from conception to execution . The women of BAPS North America share in their unwavering commitment to supporting Pramukh Swami Maharaj ’s life ’s work through advancing mahila activities . The celebration festivities brought together women of broad communities , encouraging them to reflect upon their spiritual connection to Bhagwan and pay respect to the role of a guru in strengthening that bond over time .",1,{'his': 1},{'women': 3}
248769,m,"Pottala has previously reported on a study that he led , w

In [11]:
# Warum entsteht ein Gleichstand?
examples_equal = inspect_gender_results(df, "text", case="equal")

with pd.option_context("display.max_colwidth", None):
    display(examples_equal)

# she > husband
# he > mother
#....

1,821 passende Fälle; zeige 10.


,g,text,label_text,male_terms,female_terms
334493,m,"Steve Nelson is a pastor at The Rio Church in El Paso, Texas. Steve and his wife, Kathleen, have eight children. They are passionate about raising up the next generation of on-fire believers and helping other parents do the same.Steve is the author of “Premeditated Parenting” and “31 Days of Parenting.” Tweet",2,{'his': 1},{'wife': 1}
389808,m,"Bryant Dope is a rapper/MC from Queens, New York. Bryant Dope is an MCs MC, blending intelligent and lyrically sound lines, and his own unique style to create something timelessly new. He has a couple mixtapes out including The Chronicles of Dope and Queens Kids, both of which have been securing critical acclaim. Dope is posed to leave a mark... read full bio",2,"{'his': 1, 'he': 1}",{'queens': 2}
25785,m,"Dr. Benjamin D. Sommers is an Associate Professor of Health Policy and Economics in the Department of Health Policy and Management at Harvard University’s T.H. Chan School of Public Health in Boston, MA. He is also an Associate Professor of Medicine in the Division of General Medicine and Primary Care at Brigham and Women’s Hospital/Harvard Medical School.",2,{'he': 1},{'women': 1}
97230,f,"Stephanie Moon is a photographer with a specialty in horses and pets. She enjoys both privately commissioned portrait sessions and going out into the community to create personal work. Besides here usual work of horses and pets, Stephanie also enjoys floral and landscape photography. The artist resides in Hilliard Ohio with her husband, their son, and the family dog. Awards: Betterphoto.com finalist for ""Leaning"" Betterphoto.com finalist for ""Watercolor Tulips"" Dublin Area Art League Fall 2009 Show, 3rd place ""Open Gate"" Dublin Area Art League Fall 2011 Show, 2nd place ""Winter Geese"" An Equine Jubilee: A Celebration of the Horse in Art 2012, Honorable Mention ""Teamwork"" An Equine Jubilee: A Celebration of the Horse in Art...",2,"{'husband': 1, 'son': 1}","{'she': 1, 'her': 1}"
370920,m,"Timothy Rucker is a youth pastor at Pathways Community Church in Largo, Florida, and a Th.M. student at Western Seminary in Portland, Oregon. He and his wife, Meagan, have been married since 2008 and have one daughter. Share:Click to share on Facebook (Opens in new window)Click to share on Twitter (Opens in new window)MoreClick to print (Opens in new window)Click to email this to a friend (Opens in new window)Click to share on Reddit (Opens in new window)",2,"{'he': 1, 'his': 1}","{'wife': 1, 'daughter': 1}"
380198,m,"Art Horan is an attorney who executive produced the Oscar-winning film The Usual Suspects and Ring Girls, directed by Amy Berg for ESPN’s Title IX series. He later became the head of business development at All American Heavyweights and served as EVP of global marketing for USA Boxing under the United States Olympic Committee.",2,{'he': 1},{'girls': 1}
70636,f,"Kristin Braswell is a Brooklyn-based journalist. She's contributed to ABC News, NPR and Ebony, among others. Dominique DeLeon is a filmmaker, visual artist and the Creative Director of Special Boy Films.""",2,{'boy': 1},{'she': 1}
365718,m,"Dr. Peter Spennato DDS is a famous dentist in San Pedro, California and has active license. He states that on an average an individual can visit the dentist every six months with little risk of cavities and gum disease. On the other hand however, people who are in the high risk group might need to visit every three months. The high risk group includes pregnant women, smokers, and individuals with weak immune system, oral cancer or diabetes. Moreover, Dr. Spennato states that visiting the dentist on a regular basis is also vital because certain types of medical conditions have indications that can appear in the mouth. Apart from all these one visit a dentist if any one of the below mentioned situation arise such as:",2,{'he': 1},{'women': 1}
176813,m,"Martín Espada is an acclaimed poet whose recent collections include Alabanza: New and Selected Poe

In [ ]:
# Warum wird gar kein Begriff erkannt?
examples_none = inspect_gender_results(df, "text", case="none")

with pd.option_context("display.max_colwidth", None):
    display(examples_none)

# keine Namen werden erkannt
# add Mrs. und Ms. in gender lexicon

76 passende Fälle; zeige 10.


,g,text,label_text,male_terms,female_terms
15915,f,"Mrs. Del Busto is a Counselor teacher at Curtis High School located in Staten Island, New York and part of New York City Geographic District #31. Mrs. Del Busto's 6 teacher ratings contributes to Curtis High School's ranking of 262 in New York. Additionally, the average teacher rating at Curtis High School is 4.38 stars.",3,{},{}
162719,f,"Balanced Diet Clinic is a Dietitian/Nutritionist clinic in Baner, Pune. The clinic is visited by dietitians/nutritionists like Mrs. Swapnaja Kirunge. The timings of Balanced Diet Clinic are: Mon to Sat: 11:00 AM-12:00 PM, 6:00 PM-9:00 PM. Some of the services provided by the clinic are: Pediatric Obesity Management, Weight Loss Program and Personal Diet Chart. Click on map to find directions to reach Balanced Diet Clinic.",3,{},{}
63856,f,"Mrs. Mac Lemen is a Communications teacher at Hicksville Middle School located in Hicksville, New York and part of Hicksville Union Free School District. Mrs. Mac Lemen's 25 teacher ratings contributes to Hicksville Middle School's ranking of 489 in New York. Additionally, the average teacher rating at Hicksville Middle School is 4.2 stars.",3,{},{}
9774,f,"Mrs. Grace Ebun Delano is a veteran Nurse/Midwife with an outstanding career in Reproductive Health and Family Planning management spanning over 50 years. Mrs. Delano is a Co-founder and Vice President/Executive Director of the Association for Reproductive& Family Health (ARFH), the organization that was rated as the best NGO in Nigeria. According to AIDSP AN – an independent watchdog of Global Fund and Publisher of Global Fund Observer, the organization is the 6th best performing Principal Recipient of the Global Fund project globally.",3,{},{}
209263,f,"Sara Pickett is a passionate teacher who believes in the pure joy of moving by connecting to groove and kinesthetic exploration. Mrs. Pickett's teaching history covers a wide range of subjects including all levels of modern, contemporary, jazz and hip hop techniques, dance history, choreography, improvisation, teaching methods, history of hip hop, and dance appreciation.",3,{},{}
218643,f,"Nemen Mrs is a Math teacher at Burnside High School located in Christchurch, Canterbury. When comparing Nemen Mrs' ratings to other teachers in the region of Canterbury, Nemen Mrs' ratings are below the average of 4.51 stars. Additionally, the average teacher rating at Burnside High School is 4.33 stars.",3,{},{}
306922,f,"Mrs. Rita Steff is an Accountant for the Provost's Budget Office. Mrs. Steff assists with fiscal duties relating to the Office of the Provost and Senior Vice President for Academic Affairs. These duties include all aspects of purchasing and payables, personnel and payroll, and reconciling of departmental accounts.",3,{},{}
240372,f,"Mrs. Van Camp is a teacher at J.H.S. 234 Arthur W Cunningham located in Brooklyn, New York and part of New York City Geographic District #22. Mrs. Van Camp's 4 teacher ratings contributes to J.H.S. 234 Arthur W Cunningham's ranking of 888 in New York. Additionally, the average teacher rating at J.H.S. 234 Arthur W Cunningham is 4.33 stars.",3,{},{}
227140,f,"MRS. Jamie Lynn Raymond, PA-C is a physician assistant in Charlotte, NC with 10 years of experience. The provider is a physician assistant is a person who has successfully completed an accredited education program for physician assistant, is licensed by the state and is practicing within the scope of that license. Physician assistants are formally trained to perform many of the routine, time-consuming tasks a physician can do. In some states, they may prescribe medications. They take medical histories, perform physical exams, order lab tests and x-rays, and give inoculations. Most states require that they work under the supervision of a physician. MRS. Jamie Lynn Raymond, PA-C NPI is 1164694162. The provider is registered as an individual entity type.",3,{},{}
133894,f,"Mrs. Ní Laoi is a Mathematics teacher at Cola

In [ ]:
# Welche Begriffe bleiben nach der Gender-Bereinigung übrig?
examples_remaining = inspect_gender_results(
    df, "text_without_gender", case="remaining"
)

with pd.option_context("display.max_colwidth", None):
    display(examples_remaining)

# text reinigung ist unabhängig von meinem Verfahren
# Nur subjekt entfernt

45,594 passende Fälle; zeige 10.


,g,text_without_gender,label_text_without_gender,male_terms,female_terms
68803,m,"Originally from the UK, _ has lived in Canada since 2003, and is now a dual citizen of Canada and the UK. _ is married, with two young sons. _ worked in the video games industry for 18 years (mainly for Electronic Arts on EA Sports titles) before branching out into mobile, wearables and block-chain technology, with Kitsilano Software and then doublethinkco. _ has broad-ranging technical interests and loves to write.",0,{'sons': 1},{}
8283,m,"_ research is directed toward improving the integration of genetic information into the clinical management of women with breast cancer. _ and _ colleagues have conducted a number of studies examining outcomes in women with hereditary breast cancer to better define the risks and benefits of treatments such as breast conserving therapy and adjuvant chemotherapy in this group. _ is currently conducting studies to evaluate the impact of intensive screening or surgical prevention upon women's quality of life and to develop new screening tools for breast cancer, such as serum peptide profiling.",1,{},{'women': 3}
334485,f,"_ latest book, a bilingual Indonesian-English poetry collection Racun/Poison was published in 2015 in Jakarta, Indonesia. _’s poetry has been published in various international literary journals and poetry magazines including Osiris, Ink Sweat & Tears, The Screech Owl, The Original Van Gogh’s Ear, Numéro Cinq, Deep Water Literary Journal, Fixpoetry, lyrikline.org and elsewhere as well as in anthologies like The Significant Anthology, Women in War, The Colours of Refuge, Poetry Against Racism, World Poetry Yearbook 2015, and others.",1,{},{'women': 1}
366040,m,"From May 1982 until September 1986, _. _ served as franchise and general counsel to Sbarro, Inc., the national franchisor of over 900 family-style Italian restaurants and since March 1985, _ has been a director of Sbarro, whose securities were formerly listed on the New York Stock Exchange. In addition, _ currently sits on the Board of Directors of RezConnect Technologies, Inc., Ultimate Franchise Systems, Desert Moon Café Franchise Corp., GarageTech, Inc. and Wall Street Deli Systems, Inc. From September 1983 to October 1989, _ served as President and Chairman of the Board of FranchiseIt Corporation, the first publicly traded company specializing in providing franchise marketing and consulting services and equity financing to emerging franchise companies, which _ CO-founded",0,{'chairman': 1},{}
242277,m,"Having started _ life as a classically trained violinist and knowing the obsessions that are part of the violin-world, _ became intrigued with the story of Daniel Houck, while working on a broader documentary about New vs old violins. While this movie is still in process, “Strad Style” emerged as a story that became HIS obsession.",0,{'his': 1},{}
165396,m,"In _ long and worth mentioning career, _ has embodied numerous clients in trust, estate, elder abuse cases and has got widespread experience in drafting and managing trusts and wills of people. _ has earned this reputation of being Expert in Estate Planning and Trust and Probate Law ever since 2004 and has also served as an onlooker in trust cases. _ has been familiar as a “Super Lawyer” of North California each year from 2008. _ joined _ father Edward _ and formed _ and _ Attorneys in the year 1998.",0,{'father': 1},{}
125052,f,"_ is also a member of the Ukrainian pop group, NikitA together with Yulia Kavtaradze. _ was the 2007 Playmate of the Year for the Ukrainian version of Playboy and was the American Playboy's Playmate of the Month for January 2009. _ is the 55th Anniversary Playmate, the search and selection of which were featured on episodes of The Girls Next Door. _ was also a spokeswoman for AnastasiaDate, a dating site. ... more on Wikipedia",1,{},"{'girls': 1, 'spokeswoman': 1}"
91107,f,"_ practiced in Dallas before relocating to La Jolla in 1996. Dr _ has served as Presid

: 